# J4-01 · Observabilité LLM : ce que les logs ne disent pas

**Formation « Orchestration d'agents & Monitoring : LangGraph + Langfuse (stack 100 % locale) »**
Jour 4 · créneau 08h30–09h30 · ⏱️ 1h (dont 25 min de pratique)

| | |
|---|---|
| 🎯 **Objectifs** | 1. Distinguer observabilité LLM et journalisation classique · 2. Maîtriser le modèle de données Langfuse v4 (trace, observations **typées**, session, utilisateur, score, environnement) · 3. Interroger l'**API metrics** pour obtenir les indicateurs d'un responsable Ops · 4. Lire, dans vos propres traces des jours 1 à 3, ce qui a vraiment coûté du temps |
| 🧱 **Fil rouge** | Trois jours de HelpDesk-Ops tracés : il est temps de les **mesurer** |
| ✅ **Prérequis** | Jours 1 à 3 exécutés (leurs traces sont dans votre Langfuse) |
| 📦 **Versions** | `langfuse 4.15` (SDK) · serveur Langfuse v4 |

## 1. Mise en route

### [Code] Le socle du fil rouge : `src/helpdesk_ops`

Le code écrit et expliqué en J1-01 (lecture de `.env`, masquage des données personnelles, client Langfuse, création du modèle) est rangé dans le paquet **`helpdesk_ops`**, installé par `uv sync`. Chaque notebook reste **autonome** : la cellule suivante suffit.

> 💡 **Le « Tips » du Formateur :** le paquet est installé en mode *éditable* : une modification dans `src/helpdesk_ops` est prise en compte après *Kernel → Restart*. Le client Langfuse est un singleton par clé publique : en cas de doute, redémarrez le kernel.

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

## 2. Pourquoi les logs ne suffisent pas

### [Théorie] Un bug d'agent ne lève pas d'exception

Trois jours de préparation de ce cours ont produit une collection de défauts **sans aucune erreur Python** :

| Défaut observé | Jour | Ce qu'un log aurait montré |
|---|---|---|
| ticket « TKT-2025-0012 » inventé | J3 | une réponse HTTP 200 |
| « Outlook plante » routé vers le Diagnostic | J3 | une réponse HTTP 200 |
| `[KB-010]` cité pour conseiller l'inverse de KB-010 | J3 | une réponse HTTP 200 |
| MFA traduit « authentification à facture » | J1, J3 | une réponse HTTP 200 |
| un poème écrit par l'assistant support | J3 | une réponse HTTP 200 |

| | Journalisation classique | Observabilité LLM |
|---|---|---|
| Unité | une ligne, une requête | une **trace** : l'arbre complet d'une demande |
| Défaillance | exception, code d'erreur | réponse plausible mais **fausse**, sans erreur |
| Ce qu'on regarde | statut, durée | **prompt réel**, outils et arguments, tokens, latence par étape |
| Qualité | tests unitaires | **scores**, jeux d'évaluation, retours utilisateurs |
| Variabilité | même entrée → même sortie | même question → réponses différentes (vu en J3 : PC-0203) |

## 3. Le modèle de données Langfuse v4

### [Théorie] Le ticket de caisse, complété

| Objet | Rôle | Dans HelpDesk-Ops |
|---|---|---|
| **Trace** | une demande de bout en bout | « srv-erp-02 répond-il, et le VPN ? » |
| **Observation** | une étape, **typée** : `span`, `generation`, `agent`, `tool`, `chain`, `retriever`, `evaluator`, `guardrail`, `embedding` | `coordinateur` (agent), `rechercher_procedures` (tool), `garde-fou` (guardrail) |
| **Session** | plusieurs traces d'une conversation | `j2-05-conv-julie` |
| **Utilisateur** | un pseudonyme stable | `collab-181bd0d18e` (J2-05) |
| **Score** | une note sur une trace ou une observation | 👍, `routage_juste`, `fidelite` (J4-04) |
| **Environnement** | séparer formation, recette, production | `formation` (`LANGFUSE_TRACING_ENVIRONMENT` du `.env`) |
| **Version / release** | comparer deux versions du code ou du prompt | `classifieur-v2` (défi J1-01) |

Le SDK v4 est construit sur **OpenTelemetry** : chaque observation est un *span* OTel enrichi. Ce que vous instrumentez ici se transporte vers tout outil compatible OTel.

> 💡 **Le « Tips » du Formateur : le serveur de la formation tourne en mode *events_only*.** Constaté en préparant le cours : l'API `/api/public/traces` y répond 404, on lit les données via l'API v2 des **observations** et l'API **metrics** (vues `observations` et `scores-*`). Les outils de ce jour (`helpdesk_ops.tableau_bord`, `helpdesk_ops.observabilite`) en tiennent compte.

## 4. Vos trois jours, en chiffres

L'API metrics agrège les observations côté serveur. `helpdesk_ops.tableau_bord` enveloppe les requêtes utiles.

In [ ]:
import pandas as pd

from helpdesk_ops import tableau_bord as tb

pd.set_option("display.width", 160)


def tableau(lignes: list[dict], **renommage) -> pd.DataFrame:
    df = pd.DataFrame(lignes)
    for col in [c for c in df.columns if "latency" in c]:   # conversion AVANT le renommage
        df[col] = (df[col] / 1000).round(1)                   # l'API renvoie des millisecondes
    if "traceName" in df.columns:   # observations hors d'une trace nommée (appels LLM isolés, essais)
        df["traceName"] = df["traceName"].fillna("(sans nom)")
    return df.rename(columns=renommage)


print("Volume d'observations par jour")
print(tableau(tb.volume_par_jour(langfuse), time_dimension="jour", count_count="observations").query("observations > 0").to_string(index=False))

In [ ]:
print("Où passe le temps ? Latence par nom de trace (secondes)")
print(tableau(tb.latence_par_trace(langfuse), traceName="trace", p50_latency="p50_s", p95_latency="p95_s",
              count_count="observations").sort_values("p95_s", ascending=False).head(12).to_string(index=False))

**👀 Ce que vous devez observer :** les traces multi-agents (`coordinateur`, `ex31`, `deep-agent`) dominent, de plusieurs minutes sur CPU ; les traces sans LLM (garde-fou qui refuse, sous-graphe de diagnostic) se comptent en millisecondes. C'est le premier tableau qu'un responsable Ops demande : **où** passe le temps.

In [ ]:
print("Appels LLM par modèle")
print(tableau(tb.generations_par_modele(langfuse), providedModelName="modele", count_count="appels",
              p95_latency="p95_s", sum_totalTokens="tokens").to_string(index=False))

print("\nObservations par niveau")
print(tableau(tb.erreurs_par_niveau(langfuse), count_count="observations").to_string(index=False))

**👀 Ce que vous devez observer :**
- le modèle principal concentre presque tous les appels ; sur CPU, son p95 dépasse la minute : en salle (GPU), comparez ;
- les observations de niveau **ERROR** : ce sont les tentatives en échec de J2 (retries, outil réseau instable). Ici, une erreur peut être **normale** (le retry l'a absorbée) : le niveau seul ne dit pas si l'utilisateur a été gêné.

> 💡 **Le « Tips » du Formateur : le coût d'un modèle local.** Pas de facture au token, mais du **temps GPU** : les tokens et la latence sont votre unité de coût. Si un jour le help-desk bascule sur un modèle cloud, les mêmes tableaux donneront des euros.

In [ ]:
print("Conversations les plus longues (observations par session)")
print(tableau(tb.sessions_les_plus_actives(langfuse, n=8), sessionId="session", count_count="observations").to_string(index=False))

## 5. Ce qu'un responsable Ops veut voir

| Question du responsable | Indicateur | Source |
|---|---|---|
| Le service est-il rapide ? | latence p50 / p95 par trace | `latence_par_trace` |
| Est-il fiable ? | part d'observations ERROR, replis déclenchés | `erreurs_par_niveau`, niveau des observations |
| Est-il juste ? | scores moyens (routage, fidélité, 👍) | `scores_moyens` (J4-04, J4-05) |
| Combien coûte-t-il ? | appels et tokens par modèle | `generations_par_modele` |
| Qui l'utilise ? | sessions, utilisateurs pseudonymisés | `sessions_les_plus_actives` |
| Qu'est-ce qui a changé ? | comparaison par version de prompt ou de code | versions (J4-03), experiments (J4-05) |

> ⚠️ **Une moyenne cache tout.** Un p95 de 2 minutes avec un p50 de 20 secondes dit qu'un utilisateur sur vingt attend six fois plus que les autres. Suivez toujours les **percentiles**, pas la moyenne.

## 🧭 Récap

| Vous savez maintenant… | |
|---|---|
| Pourquoi l'observabilité LLM dépasse les logs | défaillances silencieuses, prompt réel, qualité |
| Le modèle Langfuse v4 | trace, observations typées, session, utilisateur, score, environnement, version |
| Interroger les indicateurs | API metrics (`observations`, `scores-*`) via `helpdesk_ops.tableau_bord` |

### ✅ Quiz express

1. Citez deux défauts de ces trois jours qu'aucun log applicatif n'aurait signalés.
2. Pourquoi suivre le p95 plutôt que la moyenne ?
3. Une observation de niveau ERROR signifie-t-elle que l'utilisateur a été gêné ?
4. Que mesure-t-on comme « coût » avec un modèle local ?

<details>
<summary>Réponses</summary>

1. Le ticket inventé, le mauvais routage, la source contredite, la traduction fausse de MFA, le poème : tous en HTTP 200.
2. La moyenne masque la queue de distribution : le p95 dit ce que vit l'utilisateur le moins bien servi sur vingt.
3. Non : un retry ou un repli peut l'avoir absorbée. Il faut regarder la trace entière, et la réponse finale.
4. Le temps GPU : tokens et latence (et, en cloud, des euros).
</details>